# 05. Agents in LangChain 🤖

An **Agent** is a model calling tools in a loop until a task is complete.

> 💡 **Formula:** `Agent = Model + Harness` (Prompt + Tools + State)

### 1. Environment Setup & Model Initialization

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# Initialize chat model
llm = init_chat_model("groq:qwen/qwen3.8-27b")

### 2. Basic Agent with Tools (`create_agent`)
- `create_agent` is the universal factory function to build LangChain agents.

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool("addition", description="Add two numbers together")
def add(a: int, b: int) -> int:
    """Perform addition of two numbers."""
    return a + b

@tool("multiplication", description="Multiply two numbers")
def multiply(a: int, b: int) -> int:
    """Perform multiplication of two numbers."""
    return a * b

# Create the agent harness
agent = create_agent(
    model=llm,
    tools=[add, multiply],
    system_prompt="You are a helpful math expert assistant. Use the provided tools for calculations."
)

# Run the agent
response = agent.invoke({
    "messages": [{"role": "user", "content": "What is 15 multiplied by 4, and then add 10 to it?"}]
})

print(response["messages"][-1].content)

The answer is **70**.

- 15 × 4 = 60
- 60 + 10 = **70**


### 3. Agent with Structured Output (`response_format`)
- Force the agent to return a strictly validated Pydantic schema.

In [3]:
from pydantic import BaseModel, Field

class CalculationResult(BaseModel):
    user_query: str = Field(description="The original math problem requested by user")
    steps_taken: list[str] = Field(description="List of tool calls and calculations performed")
    final_answer: float = Field(description="The calculated final numeric answer")

# Create agent with response_format
structured_agent = create_agent(
    model=llm,
    tools=[add, multiply],
    system_prompt="You are a precise calculation engine.",
    response_format=CalculationResult
)

result = structured_agent.invoke({
    "messages": [{"role": "user", "content": "Calculate (25 * 4) + 50"}]
})

# Access validated Pydantic attributes
structured_output = result["structured_response"]
print("User Query:", structured_output.user_query)
print("Steps Taken:", structured_output.steps_taken)
print("Final Answer:", structured_output.final_answer)

User Query: Calculate (25 * 4) + 50
Steps Taken: ['multiplied 25 * 4 = 100', 'added 100 + 50 = 150']
Final Answer: 150.0
